# Batch Processing walkthrough (S3)

For continental / global AOIs the **Batch** plane tiles server-side and delivers to **your S3 bucket** (it needs an IAM role the Sentinel Hub service can assume). This notebook shows the request shape; it only runs end-to-end against your own bucket, so the download is guarded on both credentials **and** a configured `batch_output`.

## Credentials guard

In [ ]:
import os
from pathlib import Path


def has_sh_credentials() -> bool:
    """Whether Sentinel Hub OAuth client-credentials are available."""
    return bool(os.environ.get("SENTINELHUB_CLIENT_ID") and os.environ.get("SENTINELHUB_CLIENT_SECRET"))


def run_or_skip(facade, **download_kwargs):
    """Run the live download when credentials exist, else print a skip note.

    Keeps the notebook executing top-to-bottom with no errors whether or not
    Sentinel Hub credentials are configured (so the docs build never needs
    secrets). Set SENTINELHUB_CLIENT_ID / SENTINELHUB_CLIENT_SECRET to run the cell for real.
    """
    if not has_sh_credentials():
        print(
            "No Sentinel Hub credentials found - skipping the live download.\n"
            "Mint an OAuth client_credentials pair in the CDSE Dashboard and set\n"
            "SENTINELHUB_CLIENT_ID / SENTINELHUB_CLIENT_SECRET (see the Authentication page)."
        )
        return []
    results = facade.download(**download_kwargs)
    for item in results:
        print(item)
    return results


## Configure S3 delivery (edit for your bucket)

In [ ]:
batch_output = {
    'bucket': os.environ.get('SENTINELHUB_BATCH_BUCKET', ''),          # e.g. 's3://my-bucket/out'
    'iam_role_arn': os.environ.get('SENTINELHUB_BATCH_IAM_ROLE', ''),  # e.g. 'arn:aws:iam::123:role/sh'
    'grid_id': 2,
}
have_bucket = bool(batch_output['bucket'] and batch_output['iam_role_arn'])
print('S3 configured:', have_bucket)

## Build + submit (when both creds and a bucket are set)

In [ ]:
from earthlens import EarthLens

if has_sh_credentials() and have_bucket:
    facade = EarthLens(
        data_source='sentinel-hub',
        variables={'sentinel-2-l2a-ndvi': []},
        start='2020-06-10', end='2020-06-20',
        lat_lim=[10.0, 40.0], lon_lim=[0.0, 30.0],
        path='data/sh-batch', resolution=20,
        api='batch', batch_output=batch_output,
    )
    uris = facade.download()
    print(uris)
else:
    print('Set SENTINELHUB_CLIENT_ID / SENTINELHUB_CLIENT_SECRET and SENTINELHUB_BATCH_BUCKET / '
          'SENTINELHUB_BATCH_IAM_ROLE to run the Batch job against your bucket.')